# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yyashkumarsharma23-max/flyrank-internship-machine_learning/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import pandas as pd
import numpy as np

# 1. Loading the dataset
df = pd.read_csv('content_refresh_anonymized.csv')
playbook_df = df.copy()

# 2. Defining business logic for Actions and Reason Codes
conditions = [
    # High traffic potential but pushed past page 1 (Needs deep rewrite)
    (playbook_df['avg_position'] > 10) & (playbook_df['impressions_90d'] > 1000),
    # On page 1, but poor click-through rate (Needs title/meta fix)
    (playbook_df['avg_position'] <= 10) & (playbook_df['ctr'] < 0.02)
]

actions = ['FULL_REFRESH', 'REWRITE_METADATA']
reasons = ['HIGH_IMPACT_DECAY_RISK', 'LOW_CTR_HIGH_VISIBILITY']

# 3. Applying the labels
playbook_df['action_label'] = np.select(conditions, actions, default='NONE')
playbook_df['reason_code'] = np.select(conditions, reasons, default='NO_ACTION')

# 4. Filter and Rank the Queue (Cost/Value Strategy)
# We only want actionable items, sorted by impressions (highest value first)
action_queue = playbook_df[playbook_df['action_label'] != 'NONE'].copy()
action_queue = action_queue.sort_values(by='impressions_90d', ascending=False)

# 5. Clean output for the Content Team
final_queue = action_queue[['content_id', 'action_label', 'reason_code', 'avg_position', 'ctr', 'impressions_90d']]

print(f"Playbook Queue Generated! Total actionable URLs: {len(final_queue)}")
print("Top 10 Priority Actions for the Content Team")
display(final_queue.head(10))

Playbook Queue Generated! Total actionable URLs: 12773
Top 10 Priority Actions for the Content Team


,content_id,action_label,reason_code,avg_position,ctr,impressions_90d
19636,content_2cb567c3c89b,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,22.2,0.10,497727
29400,content_2dba2b1f9536,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,27.9,0.21,443434
26798,content_b28d1efd668f,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,26.2,0.06,286608
23767,content_813e88069237,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,26.2,0.06,233561
26304,content_ff94c9b6b411,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,27.4,0.04,228566
15968,content_66b4046cc144,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,26.6,0.03,217415
15405,content_a023517539fe,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,85.8,0.01,214047
7445,content_c8e9d6ab9013,REWRITE_METADATA,LOW_CTR_HIGH_VISIBILITY,9.7,0.00,208678
19332,content_b511d4bc4ad2,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,27.9,0.14,205915
9200,content_c5063073d048,FULL_REFRESH,HIGH_IMPACT_DECAY_RISK,12.5,0.24,192205


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Intended Use & Limits

**Intended Use (Who and For What):**

**Who uses this:** Content strategy teams, SEO managers, and editorial leads.

**For what:** This playbook serves strictly as a decision-support radar. It helps humans prioritize their limited editing hours by surfacing high-value URLs that are mathematically showing signs of decay or underperformance based on historical data.

**Limits (Where it stops being valid):**

**The 90-Day Lag:** Because the model relies on trailing 90-day metrics, it is inherently slow to react. It will not immediately detect overnight traffic drops caused by sudden Google core algorithm updates.

**The Business Value Blindspot:** The model only understands search volume and click-through rates. A low-traffic page that generates massive enterprise sales might be flagged as "low priority." The model knows SEO metrics, but it does not know your profit margins.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Human Review Rules & The No-Go List

# What a person must check before acting:

**SERP Reality Check:** Before acting on a LOW_CTR_HIGH_VISIBILITY flag, a human must look at the actual Google search results. If Google is showing a massive "Featured Snippet" or AI Overview that answers the user's question immediately, the CTR will naturally be low. Changing our title won't fix that.

**Search Intent Shift:** Before a FULL_REFRESH, an editor needs to check if the user intent has changed. (e.g., Did people stop wanting "guides" and start wanting "templates"?).

**The No-Go List (What should NEVER be automated or touched by this playbook):**

**Legal & Compliance:** Privacy policies, terms of service, and legal disclaimers.

**Core Product/Pricing Pages:** Conversion-critical pages where marketing messaging outweighs organic search volume.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Monitoring & Retrain Triggers

# How we know the recommendations are going stale:

**The Feedback Loop Failure:** If the content team executes the FULL_REFRESH on the top 50 flagged URLs, but we see less than a 20% traffic recovery after 60 days, the model's logic is no longer effective. The relationship between our features and the target has broken.

**Global CTR Drift:** If Google rolls out a major UI change, global CTRs might drop across the entire internet. If our baseline ctr < 0.02 suddenly flags 80% of our website instead of 10%, the data has drifted. We must retrain the model and adjust thresholds.

**Volume Baseline Shifts:** If the website's overall traffic grows significantly, the impressions_90d > 1000 threshold might become too low, flooding the queue with low-priority tasks.

**Time-Based Trigger:** Regardless of performance, this model should be evaluated and retrained every 6 months to capture the latest search trends and seasonality adjustments.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
import os
import matplotlib.pyplot as plt

# Repo structure ke hisaab se folders create karein (agar nahi hain)
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../outputs/charts', exist_ok=True) # Repo mein charts folder hai

# --- 1. EXPORT THE QUEUE (CSV) ---
# File ka naam wahi rakhein jo starter repo mein hai
csv_path = '../outputs/refresh_queue_sample.csv'
final_queue.to_csv(csv_path, index=False)
print(f"Ranked queue successfully exported to: {csv_path}")

# --- 2. EXPORT A VISUAL FIGURE FOR THE PAPER ---
plt.figure(figsize=(8, 5))
action_counts = final_queue['action_label'].value_counts()

# Plotting the chart
action_counts.plot(kind='bar', color=['#3498db', '#e74c3c'], edgecolor='black')
plt.title('Distribution of Recommended Content Actions', fontsize=14, fontweight='bold')
plt.xlabel('Action Type', fontsize=12)
plt.ylabel('Number of URLs', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()

# Chart ko 'charts' folder mein save karein
fig_path = '../outputs/charts/action_distribution.png'
plt.savefig(fig_path, dpi=300)
print(f"✅ Action distribution chart exported to: {fig_path}")

plt.show()

NameError: name 'final_queue' is not defined

In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.